# CPU vs GPU Performance for Machine Learning

This notebook mirrors the AWS hands-on lab, reworked with **PyTorch** on an AWS Deep Learning AMI GPU instance (e.g. `g4dn.xlarge` with an NVIDIA T4).

It walks through three experiments:

1. Confirm the GPU is visible to PyTorch
2. Matrix multiply: CPU vs GPU across sizes
3. CNN training: CPU vs GPU throughput

The goal is to *see* where the GPU helps and by how much.

## 1. Environment check

CUDA kernels run asynchronously, so any honest timing must call `torch.cuda.synchronize()` before reading the clock. We handle that throughout.

In [ ]:
import time
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('CUDA version   :', torch.version.cuda)
else:
    print('No GPU detected - results will be CPU-only.')

devices = ['cpu'] + (['cuda'] if torch.cuda.is_available() else [])
devices

In [ ]:
def synchronize(device):
    if device == 'cuda':
        torch.cuda.synchronize()

def time_op(fn, device, warmup=3, iters=10):
    """Average seconds per call. Warmup runs absorb one-time init/cache costs."""
    for _ in range(warmup):
        fn(); synchronize(device)
    start = time.perf_counter()
    for _ in range(iters):
        fn()
    synchronize(device)
    return (time.perf_counter() - start) / iters

## 2. Matrix multiply benchmark

Every dense layer in a neural net is a matrix multiply. GPUs run thousands of multiply-adds in parallel, so this is where the gap is widest at large sizes.

In [ ]:
sizes = [256, 512, 1024, 2048, 4096]
matmul_results = {d: [] for d in devices}

for device in devices:
    for n in sizes:
        a = torch.randn(n, n, device=device)
        b = torch.randn(n, n, device=device)
        secs = time_op(lambda: (a @ b).sum(), device)
        gflops = (2 * n**3) / secs / 1e9
        matmul_results[device].append((n, secs, gflops))
        print(f'[{device:4}] {n:>5} x {n:<5}  {secs*1e3:9.3f} ms   {gflops:8.1f} GFLOP/s')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
for device, rows in matmul_results.items():
    ns = [r[0] for r in rows]
    ms = [r[1] * 1e3 for r in rows]
    plt.plot(ns, ms, marker='o', label=device)
plt.xlabel('Matrix size (N x N)')
plt.ylabel('Time per matmul (ms)')
plt.title('Matrix Multiply: CPU vs GPU')
plt.yscale('log')
plt.grid(True, which='both', linestyle='--', alpha=0.4)
plt.legend()
plt.show()

## 3. CNN training benchmark

A small convolutional network trained on synthetic image-shaped data. Convolutions map extremely well to GPU hardware, so this reflects real training speedups without downloading a dataset.

In [ ]:
import torch.nn as nn

BATCH_SIZE = 64
IMAGE_SIZE = 32
NUM_CLASSES = 10

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMAGE_SIZE // 4) * (IMAGE_SIZE // 4), 256), nn.ReLU(inplace=True),
            nn.Linear(256, NUM_CLASSES),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
STEPS = 5
cnn_results = {}

for device in devices:
    model = SmallCNN().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    inputs = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    targets = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,), device=device)
    model.train()

    def train():
        for _ in range(STEPS):
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()

    secs = time_op(train, device, warmup=2, iters=5)
    per_step = secs / STEPS
    ips = BATCH_SIZE / per_step
    cnn_results[device] = ips
    print(f'[{device:4}] {per_step*1e3:8.2f} ms/step   {ips:8.1f} images/sec')

In [ ]:
plt.figure(figsize=(6, 5))
labels = list(cnn_results.keys())
vals = [cnn_results[d] for d in labels]
bars = plt.bar(labels, vals, color=['#888888', '#76b900'][:len(labels)])
for bar, v in zip(bars, vals):
    plt.text(bar.get_x() + bar.get_width()/2, v, f'{v:.0f}', ha='center', va='bottom')
plt.ylabel('Throughput (images / sec)')
plt.title('CNN Training: CPU vs GPU')
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.show()

## 4. Takeaways

- **Small matrices**: the GPU's advantage is small or even negative — kernel launch and host/device transfer overhead dominate.
- **Large matrices**: the GPU pulls far ahead as there's enough parallel work to saturate its cores.
- **CNN training**: consistently large speedups, since convolutions are compute-heavy and parallel.

The lesson for infra: GPUs pay off when the workload is big and parallel enough to keep them busy. Tiny models or tiny batches can leave a GPU idle and waste money.

> When you're done, exit the notebook and run `terraform destroy` locally to stop billing.